# TUGAS AKHIR 07 (TA-07): Implementasi dan Analisis Simulasi Sistem Dinamik
**Mata Kuliah:** Praktikum Pemodelan dan Simulasi (Modul 7)  
**Studi Kasus:** Pertumbuhan Pelanggan Platform Streaming Digital (Model Logistik)  

## 1. Formulasi Model Matematika
Pertumbuhan jumlah pengguna platform digital $P(t)$ dimodelkan menggunakan **Persamaan Diferensial Logistik**:
$$\frac{dP}{dt} = r \cdot P \left(1 - \frac{P}{K}\right)$$
Dengan komponen:
- $P(t)$: Variabel Stock (Jumlah pelanggan dalam juta jiwa)
- $r$: Laju pertumbuhan intrinsik ($0.46 \text{ tahun}^{-1}$)
- $K$: Kapasitas pasar maksimum / *carrying capacity* ($115 \text{ juta pengguna}$)

In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.integrate import solve_ivp

# Dataset Aktual Time-Series (2015 - 2025)
years = np.arange(2015, 2026)
t_data = years - 2015
users_real = np.array([12.5, 18.2, 26.0, 36.5, 49.0, 63.5, 77.0, 88.5, 97.0, 103.2, 107.5])
df_data = pd.DataFrame({'Tahun': years, 't': t_data, 'Pengguna_Aktual_Juta': users_real})
display(df_data)

In [2]:
# 2. Implementasi Simulasi Metode Euler & SciPy solve_ivp
P0 = users_real[0]
r_base = 0.46
K_base = 115.0

def logistic_ode(t, P, r, K):
    return r * P * (1 - P / K)

# Integrasi Numerik Metode Euler (h = 0.1 tahun)
h = 0.1
t_euler = np.arange(0, 10 + h, h)
P_euler = np.zeros_like(t_euler)
P_euler[0] = P0

for i in range(1, len(t_euler)):
    P_euler[i] = P_euler[i-1] + h * logistic_ode(t_euler[i-1], P_euler[i-1], r_base, K_base)

# Solver Profesional SciPy solve_ivp
sol_base = solve_ivp(logistic_ode, (0, 10), [P0], args=(r_base, K_base), t_eval=np.linspace(0, 10, 200))

# Hitung RMSE Validasi
P_pred_years = P_euler[::10]
rmse = np.sqrt(np.mean((users_real - P_pred_years)**2))
print(f'RMSE Model Euler terhadap Data Aktual: {rmse:.3f} juta pengguna')

In [3]:
# 3. Visualisasi Validasi Model
plt.figure(figsize=(9, 5))
plt.scatter(years, users_real, color='black', zorder=5, label='Data Aktual (Juta Pengguna)')
plt.plot(2015 + t_euler, P_euler, 'b--', linewidth=2, label='Simulasi Euler (h=0.1, r=0.46, K=115)')
plt.plot(2015 + sol_base.t, sol_base.y[0], 'r-', linewidth=1.8, alpha=0.8, label='Simulasi SciPy solve_ivp (RK45)')
plt.axhline(K_base, color='gray', linestyle=':', label='Carrying Capacity K = 115 Juta')
plt.title('Validasi Model Pertumbuhan Logistik: Data Aktual vs Euler & SciPy')
plt.xlabel('Tahun')
plt.ylabel('Jumlah Pengguna (Juta)')
plt.grid(True, alpha=0.4)
plt.legend()
plt.show()

In [4]:
# 4. Eksperimen Skenario What-If Analysis
sol_whatif1 = solve_ivp(logistic_ode, (0, 10), [P0], args=(r_base*0.7, K_base), t_eval=np.linspace(0, 10, 200))
sol_whatif2 = solve_ivp(logistic_ode, (0, 10), [P0], args=(r_base, 140.0), t_eval=np.linspace(0, 10, 200))

plt.figure(figsize=(9, 5))
plt.plot(2015 + sol_base.t, sol_base.y[0], 'b-', linewidth=2.2, label='Baseline (r=0.46, K=115)')
plt.plot(2015 + sol_whatif1.t, sol_whatif1.y[0], 'r--', linewidth=2, label='Skenario 1: Laju Adopsi Turun 30% (r=0.322, K=115)')
plt.plot(2015 + sol_whatif2.t, sol_whatif2.y[0], 'g-.', linewidth=2, label='Skenario 2: Ekspansi Pasar (r=0.46, K=140)')
plt.title('Analisis Skenario What-If Model Pertumbuhan Logistik')
plt.xlabel('Tahun')
plt.ylabel('Jumlah Pengguna (Juta)')
plt.grid(True, alpha=0.4)
plt.legend()
plt.show()

## 5. Interpretasi Hasil Simulasi & What-If Analysis
1. **Pola Kurva Sigmoid (S-Curve):** Pada tahun 2015–2019 ($P \ll K$), *reinforcing feedback loop* dari efek jaringan (*word-of-mouth*) mendominasi sehingga pertumbuhan berlangsung cepat. Memasuki 2021–2025 ($P \to K$), *balancing feedback loop* dari kejenuhan pasar $(1 - P/K)$ menekan laju penambahan pengguna baru.
2. **Skenario 1 (r turun 30%):** Penurunan agresivitas promosi menyebabkan titik belok (*inflection point*) tertunda sekitar 2 tahun dan total pengguna pada 2025 hanya mencapai ~94,8 juta jiwa.
3. **Skenario 2 (Ekspansi K ke 140 juta):** Pembukaan pasar regional baru meningkatkan plafon saturasi sehingga pengguna pada 2025 menembus ~127,4 juta jiwa.